# Reviewer 3 — Comment 13
## InfoNCE denominator implementation check

This notebook checks the recovered PVgraphDTA implementation used in the revision workspace, to distinguish a manuscript-equation problem from the implemented loss.

In [1]:
from pathlib import Path
import json
source_path=Path('/mnt/data/Fg-PVgraphDTA-main/src/fgpvdta/models/pvgraphdta.py')
s=source_path.read_text(encoding='utf-8')
checks={
 'cross_modal_logits':'logits = (a @ b.T) / self.temperature' in s,
 'diagonal_positive':'labels = torch.arange(a.size(0), device=a.device)' in s,
 'symmetric':'F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)' in s,
 'three_pairs':all(x in s for x in ['self._pair(ligand, protein)','self._pair(ligand, image)','self._pair(protein, image)'])
}
print(json.dumps(checks,indent=2))

{
  "cross_modal_logits": true,
  "diagonal_positive": true,
  "symmetric": true,
  "three_pairs": true
}


In [2]:
start=s.index('    def _pair')
end=s.index('    def forward', start) if '    def forward' in s[start:] else min(len(s),start+1600)
print(s[start:end])

    def _pair(self, a, b):
        logits = (a @ b.T) / self.temperature
        labels = torch.arange(a.size(0), device=a.device)
        return F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)

    def info_nce_triplet(self, ligand, protein, image):
        ligand = F.normalize(self.proj_ligand(ligand), dim=-1)
        protein = F.normalize(self.proj_protein(protein), dim=-1)
        image = F.normalize(self.proj_image(image), dim=-1)
        return self._pair(ligand, protein) + self._pair(ligand, image) + self._pair(protein, image)


